In [7]:
import cv2
import numpy as np
import time
from typing import List, Optional
from tensorflow.keras.models import Model, load_model
from landmarkers.mp.hands import MPVideoLandmarker, MediapipeHandsMetadata
from landmarkers.inferences import InferenceSequence, Inference
from landmarkers.visualization.layers import SequenceLayer, PointsLayer, BBoxLayer
from landmarkers.visualization import Viewer, ViewerBuilder
from landmarkers.visualization.visualizers import LandmarksSequenceVisualizer

# -------------------------
# Configuración
# -------------------------
ACTIONS: np.ndarray = np.array(["jump", "shoot", "none"])
SEQUENCE_LENGTH: int = 15
MODEL_EXPORT_NAME: str = "hand_gesture_model.h5"
THRESHOLD: float = 0.85
PRED_BUFFER_SIZE: int = 5
NUM_LANDMARKS: int = 21  # Cantidad de landmarks de la mano

# -------------------------
# Carga de modelos
# -------------------------
model: Model = load_model(MODEL_EXPORT_NAME)
inference_sequence: InferenceSequence = InferenceSequence(
    fixed_buffer_length=SEQUENCE_LENGTH
)

landmark_viewer: Viewer = (
    ViewerBuilder()
    .add_layer(SequenceLayer([PointsLayer(), BBoxLayer()], step=5, time_fade=True))
    .build()
)


# -------------------------
# Funciones auxiliares
# -------------------------
def get_timestamp_ms() -> int:
    """Devuelve el timestamp actual en milisegundos."""
    return int(time.time() * 1000)


def preprocess_landmarks(landmarks_seq) -> np.ndarray:
    """Resample, centrar y convertir en array lista para la red."""
    resampled = landmarks_seq.resample()
    centered = resampled.centered(0).array
    return centered.reshape(SEQUENCE_LENGTH, -1)


def predict_gesture(
    features: np.ndarray,
    model: Model,
    pred_buffer: List[np.ndarray],
    threshold: float = THRESHOLD,
) -> Optional[str]:
    """Predice el gesto y aplica suavizado con promedio de buffer."""
    res: np.ndarray = model.predict(np.expand_dims(features, axis=0), verbose=False)[0]
    pred_buffer.append(res)
    avg: np.ndarray = np.mean(pred_buffer[-PRED_BUFFER_SIZE:], axis=0)
    idx: int = int(np.argmax(avg))
    if avg[idx] > threshold and ACTIONS[idx] != "none":
        return str(ACTIONS[idx])
    return None


def render_frame(
    frame: np.ndarray,
    inference_sequence: InferenceSequence,
    landmark_viewer: Viewer,
    gesture: Optional[str] = None,
) -> np.ndarray:
    """Renderiza landmarks y gesto sobre el frame."""
    # Descomentar si quieres ver los landmarks visualizados
    # frame_out: np.ndarray = landmark_viewer.render(
    #     LandmarksSequenceVisualizer(inference_sequence.landmarks_sequence),
    #     background=frame
    # )
    if gesture:
        cv2.putText(
            frame, gesture, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3
        )
    return frame


def get_right_hand_inference(
    inferences: Optional[List[Inference]],
) -> Optional[Inference]:
    """Filtra solo la mano derecha de la lista de inferencias."""
    if inferences is None:
        return None
    right_hand: List[Inference] = [
        inf for inf in inferences if inf.metadata.category_name == "Right"
    ]
    return right_hand[0] if right_hand else None


def create_empty_right_hand_inference() -> Inference:
    """Crea una inferencia de mano derecha con todos los landmarks en (0,0,0)."""
    zeros = np.zeros((NUM_LANDMARKS, 3), dtype=np.float32)
    # Metadata mínimo para la mano derecha
    metadata: MediapipeHandsMetadata = MediapipeHandsMetadata(
        category_name="Right", index=0, score=0.0
    )

    return Inference(landmarks=zeros, world_landmarks=zeros, metadata=metadata)


# -------------------------
# Main loop con fallback de mano vacía
# -------------------------
pred_buffer: List[np.ndarray] = []

cap: cv2.VideoCapture = cv2.VideoCapture(0)

with MPVideoLandmarker(
    model_path="hand_landmarker.task", num_hands=2
) as hand_landmarker:
    while cap.isOpened():
        ret: bool
        frame: np.ndarray
        ret, frame = cap.read()
        if not ret:
            break

        timestamp: int = get_timestamp_ms()
        inferences: Optional[List[Inference]] = hand_landmarker.infer(frame, timestamp)
        right_hand: Optional[Inference] = get_right_hand_inference(inferences)
        if right_hand is None:
            right_hand = create_empty_right_hand_inference()

        inference_sequence.append(right_hand, timestamp)

        gesture: Optional[str] = None
        if len(inference_sequence) == SEQUENCE_LENGTH:
            features: np.ndarray = preprocess_landmarks(
                inference_sequence.landmarks_sequence
            )
            gesture = predict_gesture(features, model, pred_buffer)

        show_frame: np.ndarray = render_frame(
            frame, inference_sequence, landmark_viewer, gesture
        )
        cv2.imshow("Feed", show_frame)

        if cv2.waitKey(10) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1769622136.941764  161021 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769622136.955462  161021 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
